In [ ]:
#| default_exp read

## Reading and inspection

Readable notebook views plus symbol-level context for co-creation.

`read_nb` prints compact context instead of raw notebook JSON. Each cell prefix now includes dynamic classes computed from the cell source and stored outputs, so agents can quickly separate imports, exported definitions, private helpers, tests, examples, docs, headings, and mixed cells.

Use `query` when one call should answer several selections; top-level `cell_type`, `contains`, `chapter`, and `cell_id` still work as shorthand for a single selection:

```python
read_nb("nbs/01_read.ipynb", query="cell_type=exported_code contains=read_nb; cell_type=test_cell", context="overview")
read_nb("nbs/01_read.ipynb", cell_type="test_cell", context="precise")
read_nb("nbs/01_read.ipynb", contains="def read_nb", context="full")
```

The output keeps the normal cell id/source summary and adds `classes=...`, for example:

```text
Cell id=abc123: code classes=exported_code | #| export
Cell id=def456: code classes=test_cell | assert add(1, 2) == 3
```

Reading is the first problem to solve for notebook automation. An agent should not have to inspect raw `.ipynb` JSON just to learn what a notebook contains, and a human reviewer should not have to scroll through outputs and metadata to find the code.

This notebook builds compact views for three common questions: what cells exist, what full source is in a selected cell, and what documentation surrounds a symbol.

The reader is intentionally a triage tool before it is a rendering tool. A useful session usually starts with `context="overview"` to find the right cell id, switches to `context="precise"` for numbered source, and uses `context="full"` only when the surrounding rationale and examples matter.

```python
read_nb("nbs/02_write.ipynb", context="overview", show_ids=True)
read_nb("nbs/02_write.ipynb", contains="def update_cell", context="precise", show_ids=True)
read_nb("nbs/02_write.ipynb", contains="def update_cell", context="full")
```

In [ ]:
from contextlib import redirect_stdout as _redirect_stdout
from io import StringIO as _StringIO
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import mk_cell as _mk_cell, new_nb as _new_nb, write_nb as _write_nb
from nbskill.read import read_nb as _example_read_nb
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook

In [ ]:
path = demo_path("01_read_example.ipynb")
try:
    _write_nb(_new_nb([
        _mk_cell("## Demo\nWhy this function exists.", cell_type="markdown"),
        _mk_cell("#| export\ndef answer():\n    return 42", cell_type="code"),
        _mk_cell("assert answer() == 42", cell_type="code"),
    ]), path)
    _example_read_nb(str(path), context="overview", show_ids=True)
finally:
    remove_demo_path(path)

Cell id=ddb0585f hash=ac5e1b4f6955: markdown classes=docs_cell,section_header | ## Demo
Cell id=4bb06d5c hash=cd775c260999: code classes=exported_code | #| export
Cell id=91943c66 hash=c8d05d43cd97: code classes=test_cell | assert answer() == 42


In [ ]:
#| export
import ast
import copy
import json
import re
import shlex

from fastcore.nbio import read_nb as _read_nb
from fastcore.script import Param, call_parse

from nbskill.foundation import (
    cell_hash, cell_matches_type, cell_prefix, cell_source, chapter_index_set,
    cli_return, find_cell_by_id, first_line, is_definition_node,
    is_export_directive, is_exported_code_cell, matches_filter, tracked_call,
    with_context,
)

### Output shapes

The reader has to serve different attention levels. Overview mode gives a map, precise mode gives selected source with line numbers, and full mode includes nearby Markdown and example cells so a symbol has context.

In [ ]:
#| export
def _format_overview(items, show_ids=False):
    lines = []
    for idx, cell in items:
        summary = first_line(cell.source)
        lines.append(f"{cell_prefix(idx, cell, show_ids)} | {summary}")
    return "\n".join(lines)

In [ ]:
#| export
def _markdown_overview(cell):
    lines = cell.source.splitlines()
    if not any(re.match(r"^#{1,2}\s+", line.strip()) for line in lines): return []
    text = cell.source.strip()
    return [text] if text else []


def _definition_lines(node, indent=""):
    tmp = copy.deepcopy(node)
    tmp.body = [ast.Pass()]
    ast.fix_missing_locations(tmp)
    lines = []
    for line in ast.unparse(tmp).splitlines():
        if line.strip() == "pass": continue
        lines.append(f"{indent}{line}" if line else line)
    return lines


def _docstring_lines(node, indent="    "):
    doc = ast.get_docstring(node)
    if not doc: return []
    quote = indent + chr(34) * 3
    return [quote, *[f"{indent}{line}" for line in doc.splitlines()], quote]


def _function_overview(node, indent=""):
    return [*_definition_lines(node, indent=indent), *_docstring_lines(node, indent=indent + "    ")]


def _class_overview(node):
    lines = [*_definition_lines(node), *_docstring_lines(node)]
    init = next((child for child in node.body if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)) and child.name == "__init__"), None)
    if init:
        if lines: lines.append("")
        lines += _function_overview(init, indent="    ")
    return lines


def _code_overview(cell):
    try: tree = ast.parse(cell.source)
    except SyntaxError: return []
    lines = []
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)): lines += _function_overview(node)
        elif isinstance(node, ast.ClassDef): lines += _class_overview(node)
        if lines and lines[-1] != "": lines.append("")
    if lines and lines[-1] == "": lines.pop()
    return lines


def _format_headers(items, show_ids=False):
    chunks = []
    for idx, cell in items:
        if cell.cell_type == "markdown": lines = _markdown_overview(cell)
        elif cell.cell_type == "code": lines = _code_overview(cell)
        else: lines = []
        if lines: chunks.append(f"{cell_prefix(idx, cell, show_ids)}\n" + "\n".join(lines))
    return "\n\n".join(chunks)

In [ ]:
#| export
def _format_source(source, line_numbers=False):
    if not line_numbers: return source
    lines = source.splitlines() or [""]
    return "\n".join(f"{idx} | {line}" for idx, line in enumerate(lines, start=1))


def _format_full(items, show_ids=False, line_numbers=False):
    chunks = []
    for idx, cell in items:
        chunks.append(f"{cell_prefix(idx, cell, show_ids)}\n{_format_source(cell_source(cell), line_numbers=line_numbers)}")
    return "\n\n".join(chunks)

### A small query language

Automation needs stable selectors, but humans need short commands. The query helpers accept aliases like `id`, `type`, `chapter`, and `contains`, then normalize them into one internal selection shape.

In [ ]:
#| export
_QUERY_KEY_ALIASES = {
    "id": "cell_id",
    "cell": "cell_id",
    "cell_id": "cell_id",
    "chapter": "chapter",
    "type": "cell_type",
    "class": "cell_type",
    "cell_type": "cell_type",
    "contains": "contains",
    "text": "contains",
    "regex": "regex",
    "re": "regex",
}


def _normalize_query_key(key):
    name = _QUERY_KEY_ALIASES.get(str(key).strip().lower())
    if name is None:
        choices = ", ".join(sorted(_QUERY_KEY_ALIASES))
        raise ValueError(f"Unknown query key {key!r}; use one of: {choices}")
    return name


def _normalize_query_dict(spec):
    return {_normalize_query_key(key): None if value is None else str(value) for key, value in dict(spec).items()}


def _parse_query_terms(text):
    spec, bare = {}, []
    for term in shlex.split(str(text)):
        sep = "=" if "=" in term else ":" if ":" in term else None
        if sep is None:
            bare.append(term)
            continue
        key, value = term.split(sep, 1)
        spec[_normalize_query_key(key)] = value
    if bare and "contains" not in spec: spec["contains"] = " ".join(bare)
    return spec


def _parse_query(query):
    if query is None: return [{}]
    if isinstance(query, dict): return [_normalize_query_dict(query)]
    if isinstance(query, (list, tuple)):
        specs = []
        for item in query: specs.extend(_parse_query(item))
        return specs or [{}]

    text = str(query).strip()
    if not text: return [{}]
    try: parsed = json.loads(text)
    except json.JSONDecodeError:
        return [_parse_query_terms(part) for part in text.split(";") if part.strip()]
    return _parse_query(parsed)


def _merge_query(base, spec):
    merged = {key: value for key, value in base.items() if value is not None}
    merged.update({key: value for key, value in spec.items() if value is not None})
    return merged


def _query_label(spec):
    if not spec: return "all cells"
    return " ".join(f"{key}={value}" for key, value in spec.items() if value is not None)


def _select_query_items(nb, spec):
    items = [find_cell_by_id(nb.cells, spec["cell_id"])] if spec.get("cell_id") else list(enumerate(nb.cells))
    if spec.get("chapter") is not None:
        chapter_idxs = chapter_index_set(nb.cells, spec["chapter"])
        items = [(i, c) for i, c in items if i in chapter_idxs]
    if spec.get("cell_type") is not None: items = [(i, c) for i, c in items if cell_matches_type(c, spec["cell_type"])]
    if spec.get("contains") is not None: items = [(i, c) for i, c in items if spec["contains"] in c.source]
    if spec.get("regex") is not None: items = [(i, c) for i, c in items if matches_filter(c.source, spec["regex"])]
    return items


def _normalize_context(context):
    value = "overview" if context is None else str(context).strip().lower()
    if value not in {"overview", "precise", "full"}:
        raise ValueError("context must be overview, precise, or full")
    return value


def _format_context(items, context, show_ids=False):
    if context == "overview" and len(items) == 1: return _format_full(items, show_ids=show_ids, line_numbers=True)
    if context == "overview": return _format_overview(items, show_ids=show_ids)
    return _format_full(items, show_ids=show_ids, line_numbers=True)


def _cell_defined_symbols(cell):
    if getattr(cell, "cell_type", None) != "code": return []
    try: tree = ast.parse(_source_without_directives(cell_source(cell)))
    except SyntaxError: return []
    symbols = []
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)): symbols.append(node.name)
        if isinstance(node, ast.ClassDef):
            for child in node.body:
                if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)): symbols.append(f"{node.name}.{child.name}")
    return symbols


def _format_usage_for_items(path, items):
    symbols = []
    for _, cell in items: symbols.extend(_cell_defined_symbols(cell))
    if not symbols: return ""
    try:
        from nbskill.graph import symbol_usage_summary
        return symbol_usage_summary(path, symbols)
    except Exception as exc:
        return f"Usage unavailable: {type(exc).__name__}: {exc}"

### The public notebook reader

`read_nb` combines notebook loading, query selection, optional context expansion, and formatting. It is deliberately read-only, which makes it safe to use as the first tool before any edit.

In [ ]:
#| export
@call_parse
@tracked_call
def read_nb(
    path: str,  # Notebook path
    query: str | None = None,  # JSON/list or semicolon key=value clauses; combines multiple selections
    cell_id: str | None = None,  # Stable notebook cell id to select
    chapter: str | None = None,  # Chapter title string or regex to select
    cell_type: str | None = None,  # Filter cells: code/py/md/raw or dynamic classes like exported_code/test_cell
    contains: str | None = None,  # Include only cells whose source contains this text
    context: Param("overview, precise, or full", str, choices=("overview", "precise", "full")) = "overview",  # Output shape; full adds neighboring docs/examples
    show_ids: bool = False,  # Include source hashes in output
):
    "Print a compact, non-JSON view of a notebook."
    context = _normalize_context(context)

    nb = _read_nb(path)
    base = dict(cell_id=cell_id, chapter=chapter, cell_type=cell_type, contains=contains)
    specs = [_merge_query(base, spec) for spec in _parse_query(query)]
    chunks = []
    for pos, spec in enumerate(specs, start=1):
        selected = _select_query_items(nb, spec)
        items = with_context(nb.cells, selected, include=True) if context == "full" else selected
        body = _format_context(items, context, show_ids=show_ids)
        if context == "full" and (usage := _format_usage_for_items(path, selected)):
            body = f"{body}\n\nUsage:\n{usage}" if body else f"Usage:\n{usage}"
        if len(specs) == 1:
            chunks.append(body)
        else:
            label = f"Query {pos}: {_query_label(spec)}"
            chunks.append(f"{label}\n{body or '(no matches)'}")

    text = "\n\n".join(chunk for chunk in chunks if chunk)
    if text: print(text)
    return cli_return(text)

In [ ]:
#| export
def _source_without_directives(source):
    return "\n".join(line for line in source.splitlines() if not line.lstrip().startswith("#|"))

In [ ]:
#| export
def _annotation_name(annotation):
    if annotation is None: return None
    if isinstance(annotation, ast.Name): return annotation.id
    if isinstance(annotation, ast.Attribute): return annotation.attr
    if isinstance(annotation, ast.Constant): return annotation.value
    return ast.unparse(annotation)

In [ ]:
#| export
def _first_arg_annotation(node):
    args = node.args.posonlyargs or node.args.args
    return _annotation_name(args[0].annotation) if args else None

In [ ]:
#| export
def _node_defines_symbol(node, symbol):
    parts = symbol.split(".")
    name = parts[-1]
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and node.name == name:
        return True
    if len(parts) < 2: return False

    cls_name, meth_name = parts[-2], parts[-1]
    if isinstance(node, ast.ClassDef) and node.name == cls_name:
        return any(isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)) and child.name == meth_name for child in node.body)
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == meth_name:
        return _first_arg_annotation(node) == cls_name
    return False

In [ ]:
#| export
def _cell_defines_symbol(cell, symbol):
    if cell.cell_type != "code": return False
    try: tree = ast.parse(_source_without_directives(cell.source))
    except SyntaxError: return False
    return any(_node_defines_symbol(node, symbol) for node in tree.body)

In [ ]:
#| export
def _find_symbol_cell(nb, symbol):
    for idx, cell in enumerate(nb.cells):
        if _cell_defines_symbol(cell, symbol): return idx
    raise ValueError(f"Could not find symbol {symbol!r}")

In [ ]:
#| export
def _find_symbol_node(cell, symbol):
    if getattr(cell, "cell_type", None) != "code": return None
    try: tree = ast.parse(_source_without_directives(cell.source))
    except SyntaxError: return None
    parts = symbol.split(".")
    for node in tree.body:
        if len(parts) == 1 and isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and node.name == symbol:
            return node
        if _node_defines_symbol(node, symbol):
            if isinstance(node, ast.ClassDef) and len(parts) > 1:
                name = parts[-1]
                return next((child for child in node.body if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)) and child.name == name), node)
            return node
    return None

In [ ]:
#| export
def _previous_markdown(cells, idx, limit):
    docs = []
    pos = idx - 1
    while pos >= 0 and len(docs) < limit:
        cell = cells[pos]
        if getattr(cell, "cell_type", None) != "markdown": break
        docs.append((pos, cell))
        pos -= 1
    return list(reversed(docs))

In [ ]:
#| export
def _following_examples(cells, idx, limit):
    examples = []
    pos = idx + 1
    while pos < len(cells) and len(examples) < limit:
        cell = cells[pos]
        if is_exported_code_cell(cell): break
        if getattr(cell, "cell_type", None) in {"markdown", "code"}: examples.append((pos, cell))
        pos += 1
    return examples

In [ ]:
#| export
def _symbol_signature_text(cell, symbol):
    node = _find_symbol_node(cell, symbol)
    if node is None: return _code_overview(cell)
    if isinstance(node, ast.ClassDef): return _class_overview(node)
    return _function_overview(node)

In [ ]:
#| export
def _format_symbol_doc(path, nb, symbol, context=2, source=False, show_ids=False):
    idx = _find_symbol_cell(nb, symbol)
    cell = nb.cells[idx]
    lines = [f"Symbol {symbol}", "Full context: rationale/docs -> exported code -> show-off examples", cell_prefix(idx, cell, show_ids)]
    docs = _previous_markdown(nb.cells, idx, context)
    if docs:
        lines.append("")
        lines.append("Rationale/docs before the symbol:")
        for doc_idx, doc_cell in docs:
            lines.append(cell_prefix(doc_idx, doc_cell, show_ids))
            lines.append(doc_cell.source.strip())
    signature = _symbol_signature_text(cell, symbol)
    if signature:
        lines.append("")
        lines.append("Exported definition:")
        lines.extend(signature)
    if source:
        lines.append("")
        lines.append("Source cell:")
        lines.append(cell.source.strip())
    examples = _following_examples(nb.cells, idx, context)
    if examples:
        lines.append("")
        lines.append("Show-off examples after the symbol:")
        for ex_idx, ex_cell in examples:
            lines.append(cell_prefix(ex_idx, ex_cell, show_ids))
            lines.append(ex_cell.source.strip())
    if usage := _format_usage_for_items(path, [(idx, cell)]):
        lines.append("")
        lines.append("Usage:")
        lines.append(usage)
    return "\n".join(lines)

In [ ]:
#| export
@call_parse
@tracked_call
def show_doc(
    path: str,  # Notebook path
    symbol: str | None = None,  # Function, class, or Class.method to inspect
    context: int = 2,  # Rationale/docs before and show-off example cells after to include
    source: bool = False,  # Include the full source cell
    show_ids: bool = False,  # Include source hashes in output
):
    "Show rationale/docs, exported code, and show-off examples for a notebook symbol."
    if symbol is None:
        if not isinstance(path, str):
            from nbdev.showdoc import show_doc as _nbdev_show_doc
            return _nbdev_show_doc(path)
        raise ValueError("Pass symbol when path is a notebook")
    nb = _read_nb(path)
    text = _format_symbol_doc(path, nb, symbol, context=context, source=source, show_ids=show_ids)
    print(text)
    return cli_return(text)

In [ ]:
path = demo_path("01_read_doc.ipynb")
try:
    nb = new_nb([
        mk_cell("## Addition\nThis explains the exported symbol.", cell_type="markdown"),
        mk_cell("#| export\ndef add(a, b):\n    \"\"\"Add two values.\"\"\"\n    return a + b", cell_type="code"),
        mk_cell("assert add(1, 2) == 3", cell_type="code"),
    ])
    _write_nb(nb, path)
    out = _StringIO()
    with _redirect_stdout(out):
        show_doc(str(path), "add")
    text = out.getvalue()
    assert "Add two values." in text
    assert "assert add(1, 2) == 3" in text
finally:
    remove_demo_path(path)

In [ ]:
path = demo_path("01_read_sample.ipynb")
try:
    example = mk_cell("add(2, 3)", cell_type="code")
    example.outputs = [{
        "output_type": "execute_result",
        "execution_count": 1,
        "metadata": {},
        "data": {"text/plain": "5"},
    }]
    nb = new_nb([
        mk_cell("## Math\nTiny rationale.", cell_type="markdown"),
        mk_cell("#| export\ndef add(a, b):\n    \"\"\"Add values.\"\"\"\n    return a + b", cell_type="code"),
        mk_cell("assert add(1, 2) == 3", cell_type="code"),
        example,
    ])
    _write_nb(nb, path)
    out = _StringIO()
    with _redirect_stdout(out):
        show_doc(str(path), "add")
    text = out.getvalue()
    assert "rationale/docs -> exported code -> show-off examples" in text
    assert "assert add(1, 2) == 3" in text
    out = _StringIO()
    with _redirect_stdout(out):
        read_nb(str(path), context="precise", contains="def add")
    text = out.getvalue()
    assert "classes=exported_code" in text
    assert "1 | #| export" in text
    assert "2 | def add(a, b):" in text
    out = _StringIO()
    with _redirect_stdout(out):
        read_nb(str(path), context="overview", cell_type="test_cell")
    text = out.getvalue()
    assert "classes=test_cell" in text
    assert "assert add(1, 2) == 3" in text
    out = _StringIO()
    with _redirect_stdout(out):
        read_nb(str(path), context="overview", cell_type="exploration_cell")
    text = out.getvalue()
    print(text)
    assert "classes=example_cell" in text
    assert "add(2, 3)" in text
    out = _StringIO()
    with _redirect_stdout(out):
        read_nb(str(path), context="full", contains="def add")
    text = out.getvalue()
    assert "## Math" in text
    assert "assert add(1, 2) == 3" in text
    out = _StringIO()
    with _redirect_stdout(out):
        read_nb(str(path), context="overview", query="cell_type=exported_code contains=add; cell_type=test_cell")
    text = out.getvalue()
    assert "Query 1: cell_type=exported_code contains=add" in text
    assert "Query 2: cell_type=test_cell" in text
    assert "classes=exported_code" in text
    assert "classes=test_cell" in text
    out = _StringIO()
    with _redirect_stdout(out):
        read_nb(str(path), context="overview", cell_id=nb.cells[1].id)
    text = out.getvalue()
    assert "1 | #| export" in text
    assert "2 | def add(a, b):" in text
finally:
    remove_demo_path(path)

Cell id=0fe56c8d: code classes=example_cell
1 | add(2, 3)



AssertionError: 

### Symbol documentation

`show_doc` answers a different question from `read_nb`: "what should I know before changing this symbol?" These helpers find the cell that defines a function, class, or patched method, then collect the surrounding prose and examples.